# Ablation study – Credit_History
A: wszystkie cechy, B: bez historii, C: tylko historia. Identyczny podział i siatki hiperparametrów jak w pierwotnym notebooku. F1 dla Y służy do strojenia; F1 macro jest metryką dodatkową.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)

In [ ]:
df = pd.read_csv('train_u6lujuX_CVtuZ9i.csv')
X = df.drop(columns=['Loan_ID','Loan_Status'])
y = df.Loan_Status.map({'N':0,'Y':1})
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
train = X_train.assign(Loan_Status=y_train)
print('Train:', X_train.shape, '| Test:', X_test.shape)
display(pd.DataFrame({'train':y_train.value_counts(normalize=True), 'test':y_test.value_counts(normalize=True)}))

In [ ]:
display(pd.crosstab(df.Credit_History.fillna(-1), df.Loan_Status))

In [ ]:
from sklearn.base import clone
variants = {'A': list(X.columns), 'B': list(X.drop(columns='Credit_History').columns), 'C': ['Credit_History']}
num_cols_all = ['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
models = {
    'Regresja logistyczna': (LogisticRegression(max_iter=2000,random_state=RANDOM_STATE),True,
                            {'model__C':[.01,.1,1,10],'model__class_weight':[None,'balanced']}),
    'Drzewo decyzyjne': (DecisionTreeClassifier(random_state=RANDOM_STATE),False,
                         {'model__max_depth':[1,2,3,5,None],'model__min_samples_leaf':[1,5,10],
                          'model__class_weight':[None,'balanced']}),
    'Las losowy': (RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=1),False,
                   {'model__n_estimators':[100],'model__max_depth':[3,5,None],
                    'model__min_samples_leaf':[1,5,10],'model__class_weight':[None,'balanced']})
}


In [ ]:
from sklearn.metrics import confusion_matrix
rows = []
for variant, cols in variants.items():
    num_cols = [c for c in cols if c in num_cols_all]
    cat_cols = [c for c in cols if c not in num_cols_all]
    for name, (estimator, scale, grid) in models.items():
        num_steps = [('imputer', SimpleImputer(strategy='median'))]
        if scale:
            num_steps.append(('scaler', StandardScaler()))
        prep = ColumnTransformer([
            ('num', Pipeline(num_steps), num_cols),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)])
        pipe = Pipeline([('preprocessor', prep), ('model', clone(estimator))])
        search = GridSearchCV(pipe, grid, scoring={'f1':'f1','macro':'f1_macro'}, refit='f1', cv=cv, n_jobs=1, error_score='raise')
        search.fit(X_train[cols], y_train)
        pred = search.predict(X_test[cols])
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        rows.append({'Wariant':variant,'Model':name,'F1 CV':search.best_score_, 'Macro CV':search.cv_results_['mean_test_macro'][search.best_index_], 'F1 test':f1_score(y_test,pred), 'Macro test':f1_score(y_test,pred,average='macro'), 'F1 N':f1_score(y_test,pred,pos_label=0), 'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred), 'TN':tn,'FP':fp,'FN':fn,'TP':tp,'Parametry':str(search.best_params_)})
results = pd.DataFrame(rows)
display(results)
results.to_csv('ablation_results.csv',index=False)


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train,y_train)
pred = dummy.predict(X_test)
print('Dummy F1:',f1_score(y_test,pred),'Macro:',f1_score(y_test,pred,average='macro'))
for metric in ['F1 CV','F1 test','Macro test']:
    ax = results.pivot(index='Model',columns='Wariant',values=metric).plot.bar(figsize=(10,4),rot=0,ylim=(0,1),title=metric)
    ax.legend(title='Wariant'); plt.tight_layout(); plt.show()


## Interpretacja
Porównanie mierzy wartość predykcyjną cechy, nie dowodzi jej dostępności przed decyzją. Zbiór testowy był już użyty w poprzedniej analizie, więc nie jest nowym niezależnym holdoutem. F1 dla Y premiuje przewidywanie klasy większościowej; sprawdź również F1 dla N oraz macro.

## Wyniki uzyskane w wykonanym eksperymencie

| Wariant   | Model                |   F1 CV |   Macro CV |   F1 test |   Macro test |   F1 N |
|:----------|:---------------------|--------:|-----------:|----------:|-------------:|-------:|
| A         | Regresja logistyczna |  0.8714 |     0.7112 |    0.9081 |       0.8147 | 0.7213 |
| A         | Drzewo decyzyjne     |  0.8703 |     0.7074 |    0.9032 |       0.8016 | 0.7    |
| A         | Las losowy           |  0.8703 |     0.7074 |    0.9032 |       0.8016 | 0.7    |
| B         | Regresja logistyczna |  0.815  |     0.4139 |    0.8173 |       0.4087 | 0      |
| B         | Drzewo decyzyjne     |  0.814  |     0.407  |    0.8173 |       0.4087 | 0      |
| B         | Las losowy           |  0.814  |     0.407  |    0.8173 |       0.4087 | 0      |
| C         | Regresja logistyczna |  0.8703 |     0.7074 |    0.9032 |       0.8016 | 0.7    |
| C         | Drzewo decyzyjne     |  0.8703 |     0.7074 |    0.9032 |       0.8016 | 0.7    |
| C         | Las losowy           |  0.8703 |     0.7074 |    0.9032 |       0.8016 | 0.7    |

Wariant B przewiduje wyłącznie Y na zbiorze testowym. Wariant C daje identyczne wyniki jak A dla drzewa i lasu. Notebook zawiera kod do odtworzenia; obliczenia wykonano jako skrypt Python, nie przez kernel notebooka.